In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch

from statsmodels.tsa.arima.model import ARIMA
from chronos import Chronos2Pipeline
import timesfm

## Import data

In [ ]:
# from datasetsforecast.favorita import FavoritaData

# group = 'Favorita200' # 'Favorita500', 'FavoritaComplete'
# directory = './data/favorita'
# Y_df, S_df, tags = FavoritaData.load(directory=directory, group=group)


## Utils

In [ ]:
def run_experiment(series, W, H, forecast_fn):
    t          = np.arange(len(series))
    starts_idx = np.arange(W, len(series) - H)
    mae_lag, mae_caus = [], []

    for i, idx in enumerate(starts_idx):
        ctx = series[idx - W : idx]
        fut = series[idx : idx + H]

        mu, sig = ctx.mean(), ctx.std().clip(1e-6)
        fcast   = forecast_fn((ctx - mu) / sig, H)
        mae_lag.append(np.abs(fcast - (fut - mu) / sig).mean())

        mu, sig = series[:idx].mean(), series[:idx].std().clip(1e-6)
        fcast   = forecast_fn((ctx - mu) / sig, H)
        mae_caus.append(np.abs(fcast - (fut - mu) / sig).mean())

        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(starts_idx)}")

    return t, t[starts_idx], np.array(mae_lag), np.array(mae_caus)

# ── Plotting ──────────────────────────────────────────────────────────────────
def make_fig(series, t, starts, mae, path=None, dot_every=2):
    FONTSIZE = 16
    BLUE     = "#2D6B8F"
    CORAL    = "#CA6F6A"
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['font.family'] = 'serif'

    y_max   = mae.max() * 1.1
    dot_idx = np.arange(0, len(starts), dot_every)

    fig, ax1 = plt.subplots(figsize=(10, 3))
    fig.patch.set_facecolor("white")

    ax1.plot(t, series, color=BLUE, lw=1.2, alpha=0.7, zorder=2)
    ax1.set_ylabel("y (Original Space)", color=BLUE, fontsize=FONTSIZE)
    ax1.set_xlabel("t (Forecast Creation Date)", fontsize=FONTSIZE, color='black')
    ax1.set_xlim(0, len(series))
    ax1.tick_params(axis='y', labelcolor=BLUE, labelsize=FONTSIZE)
    ax1.tick_params(axis='x', labelcolor='black', labelsize=FONTSIZE)
    for sp in ax1.spines.values():
        sp.set_edgecolor("#e0e0e0")

    ax2 = ax1.twinx()
    ax2.set_ylim(0, y_max)
    ax2.set_xlim(0, len(series))
    ax2.plot(starts, mae, color=CORAL, lw=1.2, alpha=0.85, zorder=3)
    ax2.plot(starts[dot_idx], mae[dot_idx], 'o', color=CORAL, ms=3.5, zorder=4)
    ax2.set_ylabel("MAE (Norm. Space)", color=CORAL, fontsize=FONTSIZE)
    ax2.tick_params(axis='y', labelcolor=CORAL, labelsize=FONTSIZE)
    ax2.spines['right'].set_edgecolor(CORAL)
    ax2.spines['right'].set_alpha(0.5)

    plt.grid(False)
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

def summarize_errors(mae_lag, mae_caus, label=""):
    mean_lag,  std_lag  = mae_lag.mean(),  mae_lag.std()
    mean_caus, std_caus = mae_caus.mean(), mae_caus.std()
    rel = (mean_lag - mean_caus) / mean_lag

    print(f"\n{label}")
    print(f"  Lag   norm  MAE: {mean_lag:.4f} ± {std_lag:.4f}")
    print(f"  Causal norm MAE: {mean_caus:.4f} ± {std_caus:.4f}")
    print(f"  Relative reduction: {rel*100:+.1f}%")

    return dict(
        mean_lag=mean_lag, std_lag=std_lag,
        mean_caus=mean_caus, std_caus=std_caus,
        rel_reduction=rel,
    )

def run_and_plot(series, W, H, forecast_fn,
                 path_lag=None, path_caus=None, label=""):

    t, starts, mae_lag, mae_caus = run_experiment(
        series=series, W=W, H=H, forecast_fn=forecast_fn)

    summarize_errors(mae_lag, mae_caus, label=label)

    make_fig(series, t, starts, mae_lag,  path=path_lag)
    make_fig(series, t, starts, mae_caus, path=path_caus)

## ARIMA

In [ ]:
def arima_forecast(ctx_norm, h, order=(1, 1, 1)):
    fit = ARIMA(ctx_norm, order=order).fit()
    return fit.forecast(steps=h)

def forecast_fn(ctx_norm, h):
    return arima_forecast(ctx_norm, h)

# ── Synthetic ─────────────────────────────────────────────────────────────────
STEP = 5
T_TOTAL, PERIOD = 1500, 200
rng  = np.random.default_rng(42)
t_raw = np.arange(0, T_TOTAL, STEP)
series_synthetic = np.where((t_raw % PERIOD) < PERIOD // 2, 150.0, 50.0).astype(float)
series_synthetic += rng.normal(0, 2.5, T_TOTAL)[::STEP]

run_and_plot(
    series_synthetic, W=100 // STEP, H=20 // STEP,
    forecast_fn=forecast_fn,
    path_lag="./ws_lag_norm_synthetic_arima.pdf",
    path_caus="./ws_causal_norm_synthetic_arima.pdf",
)

# ── Favorita ──────────────────────────────────────────────────────────────────
df   = pd.read_csv('/home/wpotosna/forgets/figures/data/favorita/train.csv')
item = df[(df['item_nbr'] == 105574) & (df['store_nbr'] == 5)]
series_favorita = item['unit_sales'].values.astype(float)

run_and_plot(
    series_favorita, W=100, H=20,
    forecast_fn=forecast_fn,
    path_lag="./ws_lag_norm_favorita_arima.pdf",
    path_caus="./ws_causal_norm_favorita_arima.pdf")

## Chronos-2

pip install "chronos-forecasting[inference] @ git+https://github.com/amazon-science/chronos-forecasting.git"

In [ ]:
class IdentityInstanceNorm(torch.nn.Module):
    """Returns input unchanged; reports loc=0, scale=1 so de-norm is a no-op."""
    def forward(self, x, loc_scale=None):
        loc   = torch.zeros(*x.shape[:-1], 1, device=x.device, dtype=x.dtype)
        scale = torch.ones_like(loc)
        return x, (loc, scale)

    def inverse(self, x, loc_scale):
        return x  # identity: no de-norm needed
    
def chronos_forecast(pipeline, ctx_norm, h):
    # Chronos-2 needs (n_series=1, n_variates=1, history_length)
    ctx_tensor = torch.tensor(ctx_norm, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1, 1, W)
    with torch.no_grad():
        forecast = pipeline.predict(ctx_tensor, prediction_length=h)
        print(type(forecast), forecast.shape if hasattr(forecast, 'shape') else [f.shape for f in forecast])
    return forecast[0].median(dim=0).values.cpu().numpy()


pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cuda:0",
    torch_dtype=torch.bfloat16,
    )
pipeline.model.instance_norm = IdentityInstanceNorm()

def forecast_fn(ctx_norm, h):
    return chronos_forecast(pipeline=pipeline, ctx_norm=ctx_norm, h=h)

In [ ]:
# ── Synthetic ─────────────────────────────────────────────────────────────────
STEP = 5
T_TOTAL, PERIOD = 1500, 200
rng  = np.random.default_rng(42)
t_raw = np.arange(0, T_TOTAL, STEP)
series_synthetic = np.where((t_raw % PERIOD) < PERIOD // 2, 150.0, 50.0).astype(float)
series_synthetic += rng.normal(0, 2.5, T_TOTAL)[::STEP]

run_and_plot(
    series_synthetic, W=100 // STEP, H=20 // STEP,
    forecast_fn=forecast_fn,
    path_lag="./ws_lag_norm_synthetic_chronos2.pdf",
    path_caus="./ws_causal_norm_synthetic_chronos2.pdf",
)

# ── Favorita ──────────────────────────────────────────────────────────────────
df   = pd.read_csv('/home/wpotosna/forgets/figures/data/favorita/train.csv')
item = df[(df['item_nbr'] == 105574) & (df['store_nbr'] == 5)]
series_favorita = item['unit_sales'].values.astype(float)

run_and_plot(
    series_favorita, W=100, H=20,
    forecast_fn=forecast_fn,
    path_lag="./ws_lag_norm_favorita_chronos2.pdf",
    path_caus="./ws_causal_norm_favorita_chronos2.pdf"
)

## TimesFM

conda create -n timesfm_env python=3.11 -y
conda activate timesfm_env
pip install "huggingface_hub==0.23.4"
pip install timesfm
pip install numpy matplotlib pandas safetensors
pip install ipykernel

python -c "
from safetensors.torch import load_file
import torch

path = '/home/wpotosna/.cache/huggingface/hub/models--google--timesfm-2.5-200m-pytorch/snapshots/1d952420fba87f3c6dee4f240de0f1a0fbc790e3/'
weights = load_file(path + 'model.safetensors')
torch.save(weights, path + 'torch_model.ckpt')
print('done')
"

python -c "
import timesfm

model = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend='gpu',
        per_core_batch_size=32,
        horizon_len=128,
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id='google/timesfm-2.5-200m-pytorch'
    ),
)
print('OK')
"

In [ ]:
def timesfm_forecast(model, ctx_norm, h):
    point_forecast, _ = model.forecast(
        inputs=[ctx_norm],
        freq=[0],
        normalize=False,  # ← disable internal normalization
    )
    return point_forecast[0][:h]

model = timesfm.TimesFm(
        hparams=timesfm.TimesFmHparams(
            backend='gpu',
            per_core_batch_size=32,
            horizon_len=128,
        ),
        checkpoint=timesfm.TimesFmCheckpoint(
            huggingface_repo_id='google/timesfm-1.0-200m-pytorch'
        ),
    )
def forecast_fn(ctx_norm, h):
    return timesfm_forecast(model=model, ctx_norm=ctx_norm, h=h)

# ── Synthetic ─────────────────────────────────────────────────────────────────
STEP   = 5
T_TOTAL, PERIOD = 1500, 200
rng    = np.random.default_rng(42)
t_raw  = np.arange(0, T_TOTAL, STEP)
series_synthetic = np.where((t_raw % PERIOD) < PERIOD // 2, 150.0, 50.0).astype(float)
series_synthetic += rng.normal(0, 2.5, T_TOTAL)[::STEP]

run_and_plot(
    series_synthetic, W=100 // STEP, H=20 // STEP,
    forecast_fn=forecast_fn,
    path_lag="./ws_lag_norm_synthetic_timesfm.pdf",
    path_caus="./ws_causal_norm_synthetic_timesfm.pdf",
)

# ── Favorita ──────────────────────────────────────────────────────────────────
df   = pd.read_csv('/home/wpotosna/forgets/figures/data/favorita/train.csv')
item = df[(df['item_nbr'] == 105574) & (df['store_nbr'] == 5)]
series_favorita = item['unit_sales'].values.astype(float)

run_and_plot(
    series_favorita, W=100, H=20,
    forecast_fn=forecast_fn,
    path_lag="./ws_lag_norm_favorita_timesfm.pdf",
    path_caus="./ws_causal_norm_favorita_timesfm.pdf"
)